# 3. Outfit generation check
This notebook inspects configuration sampling, candidate coverage, candidate images, and—after indexing—top recommendations for one product. Generated candidates contain products excluded from selected training outfits.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config.yml').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.setup_utilities import load_config
from src.data_processing.schema import PRODUCT_CLASSES
config = load_config(PROJECT_ROOT / 'config.yml')
print('Project:', PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rec = config['recommendation']
candidates = pd.read_csv(rec['candidates_path'], dtype=str, keep_default_na=False)
frequencies = pd.read_csv(rec['configurations_path'])
used_products = {product for slot in PRODUCT_CLASSES for product in candidates[slot] if product}
display(pd.Series({
    'candidate draws': len(candidates),
    'distinct outfits': candidates.signature.nunique(),
    'covered products': len(used_products),
    'allowed configurations': len(frequencies),
    'observed configurations': int((frequencies.training_count > 0).sum()),
}, name='count').to_frame())
observed = frequencies[frequencies.training_count > 0].copy()
observed['sample_probability'] = observed.sampled_count / observed.sampled_count.sum()
observed.plot(x='configuration_id', y=['probability', 'sample_probability'], kind='bar', figsize=(16, 5))
plt.title('Training probability versus sampled probability')
plt.ylabel('Probability')
plt.tight_layout()

Choose a candidate row below. Change `CANDIDATE_ROW` and rerun this cell to inspect another generated outfit.

In [ ]:
from PIL import Image

CANDIDATE_ROW = 0
products = pd.read_csv(config['data']['products_path'], dtype={'cod_modelo_color': str})
image_lookup = products.set_index('cod_modelo_color').des_filename.map(lambda value: PROJECT_ROOT / 'data/images' / Path(value).name).to_dict()

def show_outfit(product_map, title):
    present = [(slot, str(product_map[slot])) for slot in PRODUCT_CLASSES if product_map.get(slot)]
    fig, axes = plt.subplots(1, len(present), figsize=(2.3 * len(present), 3.4), squeeze=False)
    for axis, (slot, product_id) in zip(axes[0], present):
        axis.imshow(Image.open(image_lookup[product_id]).convert('RGB'))
        axis.set_title(f'{slot}\n{product_id}', fontsize=9)
        axis.axis('off')
    fig.suptitle(title)
    plt.tight_layout()

candidate = candidates.iloc[CANDIDATE_ROW]
show_outfit(candidate.to_dict(), f"Generated {candidate.candidate_id}; configuration {candidate.configuration_id}")

After `build_index` completes, choose an input product. The lookup returns the lowest-distance distinct candidates containing that exact product. Lower distance means closer to a selected training outfit embedding; it is not a probability.

In [ ]:
from src.recommendation.ranking import recommend

PRODUCT_ID = next(product for product in candidate[list(PRODUCT_CLASSES)] if product)
index_path = Path(rec['index_path'])
if index_path.is_file():
    recommendations = recommend(index_path, PRODUCT_ID, top_k=3)
    display(pd.DataFrame([{key: value for key, value in item.items() if key != 'products'} for item in recommendations]))
    for rank, item in enumerate(recommendations, start=1):
        show_outfit(item['products'], f"Rank {rank}; distance={item['distance']:.4f}; nearest training outfit={item['nearest_training_outfit']}")
else:
    print('Recommendation index is not available. Run: python -m scripts.run_pipeline --from-stage index --to-stage index')